# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing all schema elements by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Dataset Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets and their fields, referenced by their `@id`.

In [ ]:
# List out all record sets and their fields using their @id
from pprint import pprint

record_sets = list(dataset.record_sets())

print(f"Total Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set name: {rs.name}")
    print(f"Record Set @id: {rs.id}")
    print("Field @ids:")
    for field in rs.fields:
        print(f"  - {field.id} ({field.name})")
    print()

## 3. Data Extraction
Load data from each record set into DataFrames for further analysis. All references use the `@id` of each entity.

In [ ]:
# Extract data from each record set by their @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    # records returns dicts where the keys are field @ids
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id} (shape={df.shape})")
    else:
        print(f"No records found for record set {record_set_id}")

# For demonstration, print columns of the first non-empty DataFrame
for recset_id, df in dataframes.items():
    print(f"\nColumns for Record Set {recset_id}:")
    print(list(df.columns))
    print(df.head())
    break  # Only show for the first one

## 4. Exploratory Data Analysis (EDA)
We now demonstrate numeric processing using field and record set `@id`s as required by the Croissant schema.

In [ ]:
# Choose a record set and a numeric field for example processing
# Adjust these IDs based on the printed overview if necessary:
selected_record_set_id = next(iter(dataframes.keys()))  # Pick the first data-loaded record set
df = dataframes[selected_record_set_id]

# Let's auto-select a likely numeric field (@id ending with 'MW', 'Abundance', 'coverage', or 'count')
numeric_field_id = None
possible_patterns = ['MW', 'abundance', 'Abundance', 'coverage', 'count', 'peptide', 'Amount']
for col in df.columns:
    if any(p.lower() in col.lower() for p in possible_patterns):
        if pd.api.types.is_numeric_dtype(df[col]) or df[col].astype(str).str.replace('.', '', 1).str.isnumeric().any():
            numeric_field_id = col
            break

if numeric_field_id is None:
    # If not auto-found, just pick the first column
    numeric_field_id = df.columns[0]

print(f"Using field @id for numeric analysis: {numeric_field_id}")

# Try to ensure the selected column is numeric
try:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
except Exception:
    pass

threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fi' else 0

# Filtering
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (total: {len(filtered_df)}):")
print(filtered_df.head())

# Normalization
if filtered_df[numeric_field_id].std() != 0:
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a possible categorical (for demonstration, pick first textual)
group_field_id = None
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
        group_field_id = col
        break

if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df[[numeric_field_id]].head())

## 5. Visualization
Visualize the numeric field's distribution and any relationships to a categorical grouping where available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot by group field if available
if group_field_id is not None:
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, you explored the FAIR^2 Mass Spectrometry dataset with `mlcroissant`, loading and processing data entirely via entity `@id`. You:

- Loaded dataset metadata and inspected the schema structure.
- Extracted data from record sets and referenced all fields by `@id`.
- Performed basic EDA, including filtering, normalization, and grouping using field identifiers.
- Visualized numeric distributions and categorical differences.

This workflow can be adapted to other Croissant-standard datasets: all transformations and references remain compatible and reproducible due to the stable usage of semantic entity identifiers.